# Predicting the Electronic Band Gap of Inorganic Crystalline Materials from Composition and Structure
### Stage 3 — Environment Setup and Data Acquisition

This notebook implements **Stage 3 only**: setting up a reproducible Google Colab environment and acquiring the raw dataset from the Materials Project.

**No modeling, EDA, or feature engineering happens in this notebook.** It ends with a raw data export and a data-quality report.

**Correction applied from the Stage 2 plan:** the previously proposed `energy_above_hull <= 0.2 eV/atom` stability filter is **not** applied here. We acquire the raw data first, inspect the actual `energy_above_hull` distribution, and decide in Stage 4 whether any cutoff is scientifically justified — with the exact threshold, records removed, and rationale documented at that point.


## Step 3A — Environment Setup

We install only the packages this stage actually needs: `mp-api` (Materials Project client), `pymatgen` (its dependency, also used for composition/structure objects later), and the standard scientific-Python stack. We print every package's exact version and set a random seed, so the environment this ran in is fully documented and reproducible.


In [ ]:
# Step 3A.1 — Install required packages
# Run this once per Colab session. Pinning is intentionally avoided here so we
# always get the current client; the exact installed versions are printed and
# recorded in Step 3H so the snapshot is still fully reproducible.
!pip install -q mp-api pymatgen pandas numpy matplotlib scikit-learn scipy seaborn


In [ ]:
# Step 3A.2 — Imports, version reporting, and random seed
import sys
import platform
import random
import json
import importlib.metadata as importlib_metadata
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import mp_api
import pymatgen
from mp_api.client import MPRester

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# mp_api itself does not expose a __version__ attribute (confirmed by the
# AttributeError this raised) -- installed-package metadata is the reliable
# way to get it. pymatgen does define __version__, but we use the same
# metadata approach for both so this doesn't silently break again if that changes.
MP_API_VERSION = importlib_metadata.version("mp-api")
PYMATGEN_VERSION = importlib_metadata.version("pymatgen")

print("Python version:      ", sys.version.split()[0])
print("Platform:             ", platform.platform())
print("mp-api version:      ", MP_API_VERSION)
print("pymatgen version:    ", PYMATGEN_VERSION)
print("pandas version:      ", pd.__version__)
print("numpy version:       ", np.__version__)
print("Random seed set to:  ", RANDOM_SEED)


## Step 3B — Materials Project API Access

The API key is **never hard-coded**. Two options are shown below:

1. **`getpass`** — works in any environment (Colab, local Jupyter, plain Python script). The key is entered at runtime, held only in memory for the session, and never written to disk or printed.
2. **Colab Secrets (`google.colab.userdata`)** — Colab's built-in secrets manager. You add the key once via the key icon in the left sidebar, and it's injected at runtime without ever appearing in the notebook's cells or output. This is preferable if you'll re-run this notebook many times, since you won't have to retype the key each session — but it ties the notebook to Colab specifically, whereas `getpass` works anywhere.

Either way: the key is **never printed**, and this notebook must never be committed to GitHub with a key pasted into a cell or its output.


In [ ]:
# Step 3B — Provide the Materials Project API key at runtime (choose ONE)

USE_COLAB_SECRETS = False  # set True if running in Colab with a secret named MP_API_KEY

if USE_COLAB_SECRETS:
    from google.colab import userdata
    MP_API_KEY = userdata.get("MP_API_KEY")
else:
    from getpass import getpass
    MP_API_KEY = getpass("Enter your Materials Project API key (input hidden): ")

assert MP_API_KEY, "No API key provided — get one free at https://next-gen.materialsproject.org/api"
print("API key received (length: {} characters). Not printed.".format(len(MP_API_KEY)))


## Step 3C — Verify Currently Available Fields

Materials Project's schema can change between client releases, so we don't trust field names from documentation alone — we ask the live client what it currently exposes, and cross-check that against every field this project plans to use.


In [ ]:
# Step 3C — Inspect the fields actually exposed by the installed mp-api client
with MPRester(MP_API_KEY) as mpr:
    available = mpr.materials.summary.available_fields

print(f"Total fields exposed by materials.summary endpoint: {len(available)}\n")
for f in available:
    print(" -", f)

PLANNED_FIELDS = [
    "material_id", "formula_pretty", "formula_anonymous", "chemsys",
    "elements", "nelements", "composition", "composition_reduced",
    "band_gap",
    "nsites", "volume", "density", "density_atomic", "symmetry",
    "formation_energy_per_atom", "energy_above_hull", "is_stable",
    "deprecated",
]

missing = [f for f in PLANNED_FIELDS if f not in available]
if missing:
    print("\n*** WARNING: the following planned fields are NOT in the current schema:", missing)
    print("*** Do not silently substitute a similar-sounding name — investigate the schema change first.")
else:
    print("\nAll planned fields confirmed present in the current mp-api schema.")


## Step 3D — Small Test Query

Before touching the full population, we pull a tiny sample (20 records) to inspect the returned data types, confirm the composition and structure representations, and check that the result converts cleanly into a pandas DataFrame without losing information. This run **does** request the full `structure` object, purely so we can see what it looks like — the bulk pull in Step 3G deliberately will not (see the note there).


In [ ]:
# Step 3D — Test query: 20 records, including the full structure object for inspection only
TEST_FIELDS = PLANNED_FIELDS + ["structure"]

with MPRester(MP_API_KEY) as mpr:
    test_docs = mpr.materials.summary.search(
        num_elements=(1, 9),         # broad, explicit filter — see Step 3E note on why we always pass a filter
        fields=TEST_FIELDS,
        num_chunks=1,
        chunk_size=20,
    )

print(f"Retrieved {len(test_docs)} test records.\n")

example = test_docs[0]
print("Example material_id:      ", example.material_id)
print("Example formula:          ", example.formula_pretty)
print("Example band_gap (eV):    ", example.band_gap)
print("Example composition type: ", type(example.composition))
print("Example structure type:   ", type(example.structure))
print("Example symmetry object:  ", example.symmetry)


In [ ]:
# Step 3D continued — confirm clean conversion to a DataFrame (scalar fields only;
# the full Structure object is kept aside separately since it isn't tabular)
def doc_to_row(doc):
    sym = doc.symmetry
    return {
        "material_id": str(doc.material_id),
        "formula_pretty": doc.formula_pretty,
        "formula_anonymous": doc.formula_anonymous,
        "chemsys": doc.chemsys,
        "nelements": doc.nelements,
        "elements": [str(e) for e in doc.elements] if doc.elements else None,
        "band_gap": doc.band_gap,
        "nsites": doc.nsites,
        "volume": doc.volume,
        "density": doc.density,
        "density_atomic": doc.density_atomic,
        "crystal_system": str(sym.crystal_system) if sym and sym.crystal_system else None,
        "space_group_symbol": sym.symbol if sym else None,
        "space_group_number": sym.number if sym else None,
        "formation_energy_per_atom": doc.formation_energy_per_atom,
        "energy_above_hull": doc.energy_above_hull,
        "is_stable": doc.is_stable,
        "deprecated": doc.deprecated,
    }

test_df = pd.DataFrame([doc_to_row(d) for d in test_docs])
print(test_df.shape)
test_df.head(10)


## Step 3E — Define the Initial Population

**No filtering on the target.** `band_gap == 0` (metallic) materials are kept — they are scientifically part of this problem (the metal/insulator boundary is one of the things we want to study). No restriction to transition metals, oxides, or semiconductors.

**Important, verified caveat about `mp-api` query behavior:** community reports on the Materials Project support forum document that calling `.summary.search()` with **no query parameters at all** routes the request through a different backend (a bulk AWS Open Data path) that returns a *different, larger* record set than the normal filtered path — including deprecated entries and separately-licensed GNoME-derived materials that are not part of the standard queryable database. To stay on the normal, filtered path, we always pass at least one real filter. `num_elements=(1, 9)` is used here specifically because it is broad enough to be scientifically neutral (it does not exclude any chemically realistic inorganic compound) while still routing the query correctly.

The only exclusions applied at this stage are basic QA, not chemistry or target-based:
- `deprecated == True` records are excluded (MP-flagged superseded calculations)
- Records missing `band_gap` are excluded (nothing to predict)
- Records missing the structural scalar fields we plan to use are flagged and counted, not silently dropped


In [ ]:
# Step 3E — Query parameters for the full population (defined here, executed in Step 3F/3G)
QUERY_PARAMS = dict(
    num_elements=(1, 9),   # broad, explicit filter — keeps the query on the normal filtered path (see note above)
)

# Deliberately excluded from the bulk pull: the full pymatgen `structure` object.
# All of the structural descriptors planned for this project (nsites, volume, density,
# density_atomic, crystal_system, space_group) are already available as lightweight
# scalar/summary fields — pulling the full atomic-position Structure for every one of
# tens of thousands of records would multiply payload size for no planned benefit.
# If a later stage needs full structures (e.g. for coordination-environment descriptors),
# they can be fetched for a specific subsample by material_id at that point.
BULK_FIELDS = PLANNED_FIELDS  # i.e. PLANNED_FIELDS without "structure"
print("Query parameters:", QUERY_PARAMS)
print("Fields requested:", BULK_FIELDS)


## Step 3F — Determine Dataset Size Before Full Download

We first request only `material_id` (the lightest possible payload) under the same query parameters, purely to see how many records we're dealing with before pulling every field for every record.


In [ ]:
# Step 3F — Lightweight size check
with MPRester(MP_API_KEY) as mpr:
    id_only_docs = mpr.materials.summary.search(
        **QUERY_PARAMS,
        fields=["material_id", "deprecated", "band_gap"],
    )

total_returned = len(id_only_docs)
n_deprecated = sum(1 for d in id_only_docs if d.deprecated)
n_missing_bandgap = sum(1 for d in id_only_docs if d.band_gap is None)
n_after_basic_qa = sum(
    1 for d in id_only_docs if (not d.deprecated) and (d.band_gap is not None)
)

print(f"Total records returned by query:        {total_returned}")
print(f"Deprecated records:                     {n_deprecated}")
print(f"Records with missing band_gap:          {n_missing_bandgap}")
print(f"Records remaining after basic QA:       {n_after_basic_qa}")


## Step 3G — Retrieve and Save the Raw Dataset

Now we pull all planned scalar fields for the full population (still excluding the full `structure` object, per the Step 3E note). This becomes the raw, unmodified dataset — no filtering beyond the basic QA already reported above.

We save:
1. The raw CSV itself (kept locally in Colab / your own storage — **not** committed to GitHub if it turns out to be large; the repo will instead hold this notebook plus a standalone acquisition script so anyone with their own API key can regenerate it)
2. A small sample CSV (first 200 rows) that *can* safely go in the repo for a quick look without needing an API key
3. A JSON metadata sidecar recording exactly how this data was obtained (Step 3H)


In [ ]:
# Step 3G — Full data pull (scalar/summary fields only, no target filtering)
with MPRester(MP_API_KEY) as mpr:
    all_docs = mpr.materials.summary.search(
        **QUERY_PARAMS,
        fields=BULK_FIELDS,
    )

print(f"Retrieved {len(all_docs)} full records.")

raw_df = pd.DataFrame([doc_to_row(d) for d in all_docs])
print(raw_df.shape)
raw_df.head()


In [ ]:
# Step 3G continued — save raw data, sample, and metadata
import mp_api as _mp_api_pkg  # already imported above; re-referenced for clarity

RAW_CSV_PATH = "mp_bandgap_raw.csv"
SAMPLE_CSV_PATH = "mp_bandgap_raw_sample.csv"
METADATA_PATH = "mp_bandgap_acquisition_metadata.json"

raw_df.to_csv(RAW_CSV_PATH, index=False)
raw_df.head(200).to_csv(SAMPLE_CSV_PATH, index=False)

print(f"Saved full raw dataset to:   {RAW_CSV_PATH}  ({raw_df.shape[0]} rows)")
print(f"Saved repo-safe sample to:   {SAMPLE_CSV_PATH}  (200 rows)")


## Step 3H — Record the Database Snapshot

Materials Project is a **continuously updated** database — new calculations are added and existing ones are occasionally revised. Without recording the exact retrieval date and client versions, this dataset cannot be regenerated identically later, which would break reproducibility. This metadata file is what makes Step 3G's raw CSV meaningful months from now.


In [ ]:
# Step 3H — Save acquisition metadata
metadata = {
    "retrieval_datetime_utc": datetime.now(timezone.utc).isoformat(),
    "mp_api_version": MP_API_VERSION,
    "pymatgen_version": PYMATGEN_VERSION,
    "python_version": sys.version.split()[0],
    "query_parameters": QUERY_PARAMS,
    "fields_requested": BULK_FIELDS,
    "filtering_applied": {
        "deprecated_excluded": True,
        "missing_band_gap_excluded": False,  # see note below
        "energy_above_hull_cutoff": None,    # deliberately NOT applied yet — see Stage 3 intro note
    },
    "n_records_retrieved_before_qa": total_returned,
    "n_deprecated_excluded": n_deprecated,
    "n_missing_band_gap": n_missing_bandgap,
    "n_records_in_raw_csv": raw_df.shape[0],
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))


*Note on `missing_band_gap_excluded: False` above:* the raw CSV saved in Step 3G is the **unfiltered** pull (only `deprecated` records were excluded during the Step 3F count, but the Step 3G bulk pull itself applies no exclusions at all beyond the query parameters) — records with a missing `band_gap`, if any exist, are still present in `mp_bandgap_raw.csv` and will be explicitly counted and handled in the data-quality report below and in Stage 4, not silently dropped during acquisition.


## Step 3I — Initial Data Quality Report

This is a description of the raw data as retrieved — **no modeling conclusions are drawn here.**


In [ ]:
# Step 3I — Data quality report
report = {}

report["n_records_retrieved"] = int(raw_df.shape[0])
report["n_deprecated"] = int(raw_df["deprecated"].sum())
report["n_missing_band_gap"] = int(raw_df["band_gap"].isna().sum())
report["n_with_band_gap"] = int(raw_df["band_gap"].notna().sum())
report["n_metallic_bandgap_zero"] = int((raw_df["band_gap"] == 0).sum())
report["n_positive_band_gap"] = int((raw_df["band_gap"] > 0).sum())

missing_summary = raw_df.isna().sum()
report["missing_values_by_column"] = {k: int(v) for k, v in missing_summary.items() if v > 0}

report["band_gap_descriptive_stats"] = raw_df["band_gap"].describe().to_dict()
report["energy_above_hull_descriptive_stats"] = raw_df["energy_above_hull"].describe().to_dict()

# Transition-metal-containing count (d-block elements; used for reporting only, not for filtering)
D_BLOCK = set([
    "Sc","Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn",
    "Y","Zr","Nb","Mo","Tc","Ru","Rh","Pd","Ag","Cd",
    "Hf","Ta","W","Re","Os","Ir","Pt","Au","Hg",
])
report["n_transition_metal_containing"] = int(
    raw_df["elements"].apply(lambda els: bool(set(els) & D_BLOCK) if isinstance(els, list) else False).sum()
)

report["n_unique_chemsys"] = int(raw_df["chemsys"].nunique())
all_elements = set()
raw_df["elements"].dropna().apply(lambda els: all_elements.update(els) if isinstance(els, list) else None)
report["n_unique_elements"] = len(all_elements)

print(json.dumps(report, indent=2, default=str))

with open("mp_bandgap_stage3_quality_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)


### Stage 3 ends here.

No stability filtering, feature engineering, EDA visualizations, or modeling has been performed. The next stage (Stage 4) will use the `energy_above_hull` distribution and the missing-value/QA numbers printed above to decide, with justification, whether any stability cutoff is warranted — and will report the exact threshold, records removed, and percentage removed if one is applied.

**Do not run Stage 4 code in this notebook.** Await confirmation before proceeding.
